# Installation

## SWI-Prolog

SWI-Prolog is required to run LangPro. During the installation you will have to press ENTER to continue installing swi-prolog.

In [ ]:
#!sudo apt-get install software-properties-common
!sudo apt-add-repository -y ppa:swi-prolog/stable
!sudo apt-get update
!sudo apt-get install swi-prolog

In [ ]:
# test whether swi-prolog is installed and check its version
! swipl --version

## LangPro
Natural Tableau-based theorem prover that can operate on parsed sentences and detect semantic relations between a set of premises and a hypothesis. While creating this notebook, the `nl` branch of the LangPro repo is most up to date and stable.

In [ ]:
#! git clone https://github.com/kovvalsky/LangPro.git
#! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git
# just to make sure it points the specific commit on which the notebook was tested
#! cd LangPro; git reset --hard dfc0a00f46240e80675139089afdb82cab112332
#! cd LangPro; git pull
! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git

## C&C tools
C&C tools ([Clark&Curran, 2007](https://www.aclweb.org/anthology/J07-4004.pdf)) contain POS tagger, named-entity recognizer (NER), CCG parser, and Boxer. We need only the POS tagger, NER and parser. If the data doesn't contain named entities, the NER is irrelevant. It comes with only with the 🇬🇧 English models. As an input, LangPro requires the prolog format of the CCG derivation trees (hence, `--candc-printer boxer`).

In [ ]:
! git clone https://github.com/chrzyki/candc.git

In [ ]:
!candc/candc/bin/candc --version

In [ ]:
! tar -xzf candc/models/models-1.02.tgz -C candc/models

In [ ]:
# test that C&C tools (namely, POS tagger, NER and parser) are working
! echo "Parse this sentence for me" | candc/candc/bin/candc --models candc/models/models --candc-printer boxer

# SICK data

Download SICK (Sentences Involving Compositional Knowledge) data ([Marelli et al. 2014](http://www.lrec-conf.org/proceedings/lrec2014/pdf/363_Paper.pdf)). You can also mount the google drive with the data on it and adjust the parsing scripts. Note that the dataset is downloaded from [SemEval 2014 task-1](https://aclanthology.org/S14-2001/) because this is the version (in total 9927 problems) that is usually used as a benchmark (rather than other version of SICK which misses ~84 problems).

In [ ]:
! wget -P SICK \
http://alt.qcri.org/semeval2014/task1/data/uploads/sick_trial.zip \
http://alt.qcri.org/semeval2014/task1/data/uploads/sick_train.zip \
http://alt.qcri.org/semeval2014/task1/data/uploads/sick_test_annotated.zip

In [ ]:
! unzip -o SICK/\*.zip -d SICK/

## Preparing Data for LangPro

Langpro requires two prolog files per NLI dataset/part. `*_sen.pl` file records NLI problems with labels in a prolog format. `*_ccg.pl` file contains parse trees of NLI sentecnes. Using both files, for the $N$th NLI problem, LangPro looks up the problem's sentence IDs in `*_sen.pl` and then finds the sentence parse trees in `*_ccg.pl` based on the IDs.

It is recommended that you once obtain `*.pl` files and use them for later proving. You don't want to parse the sentences each time you trigger LangPro. This save a lot of runtime. Here, we are creating required `*.pl` files.

For each SICK part, create a file with sentences per line, used for parsing all sentences at once. But before parsing the sentences, we need to be tokenized them first as the parsers expect a tokenized input. In this example we will use NLTK's tokenizer (the script also support SpaCy's tokenizer). That's why we need to make sure that the tokenizer works.

In [ ]:
# required for the NLTK tokenizer
import nltk
nltk.download('punkt')

In [ ]:
# A directory where the prolog files necessary for theorem proving is written
! mkdir SICK_pl

In [ ]:
# Convert the NLI data into a prolog format (*_sen.pl), used by LangPro for reporting raw problems
! for f in `ls SICK/SICK_*.txt | xargs -n 1 basename`; do\
    python3 LangPro/python/nlidata2prolog.py  SICK/$f  SICK_pl/${f/.txt/_sen.pl} \
    --fmt sen.pl  --tokenize nltk  --corpus sick_semeval; done
# Problems read & converted should be: 500 + 4500 + 4927

In [ ]:
# tokenize NLI sentences and write in a sentence-per-line format to be parsed later
! for f in `ls SICK/SICK_*.txt | xargs -n 1 basename`; do\
    python3 LangPro/python/nlidata2prolog.py  SICK/$f  SICK/${f/txt/spl} \
    --fmt spl  --tokenize nltk  --corpus sick_semeval; done

Parsing ths sentences with C&C tools. After parsing you might want to save the obtained `*_ccg.pl` files because you want to parser the sentences once, not every time the prover starts reasoning.

In [ ]:
! for f in `ls SICK/SICK_*.spl | xargs -n 1 basename`; do\
    cat SICK/$f | \
    candc/candc/bin/candc --models candc/models/models --output SICK_pl/${f/.spl/_cc_ccg.pl} \
    --candc-printer boxer --candc-parser-noisy_rules=false; done
# give a minute to parse all ~10K sentences

Now we already have all necessary prolog files in `SICK_pl` for reasoning with LangPro.  
Note that if some sentence is not parsed, its corresponding `ccg($id,...` term won't be in `*_ccg.pl` file. For example, in `SICK_train_cc_ccg.pl` we have only 8997 CCG derivations while there should be 9000 as the train part contains 4500 problems and $2 \times 4500$ sentences (i.e., a problem consists of a single premise and a hypothesis). The missing 3 derivations is not necessarily for three different sentences as the sentences repeat in the SICK problems.

In [ ]:
! grep -P "ccg\(\d+" SICK_pl/SICK_train_cc_ccg.pl | wc -l

# NLI Proving

The elements of `parList` are described [here](https://github.com/kovvalsky/LangPro/wiki/Using-the-prover). The ones you might want to change are:
* `ral(50)` - rule application limit, which means that less you set there less time will be spend to find a proof and the results might be poor);
* `waif(filename)` - write answers in file. In case you want to have LangPro predictions in a file written.;
* `prprb` - by default LangPro prints problems that were not predicted correctly. This flag forces LangPro to print all the problems.

## SICK proving

When running LangPro, we need to feed it with wordnet files to give it access to some lexical knowledge. Other files that needs to be given are prolog files with NLI problem descriptions and parses.

In [ ]:
# Directory where NLI proving judgements will be written
! mkdir SICK_proving

In [ ]:
# Proving the NLI problems from the trial part of SICK (i.e. 500 problems)
! swipl -g "parList([prprb, ral(50), allInt, aall, wn_ant, wn_sim, wn_der, constchk, waif('SICK_proving/SICK_trial_pred_50.txt')]), entail_all, halt" \
-f  LangPro/prolog/main.pl  LangPro/WNProlog/wn.pl  SICK_pl/SICK_trial_sen.pl  SICK_pl/SICK_trial_cc_ccg.pl
# printed information should be interpreted as:
# problem_D: [gold_label], predicted_label, theorem_proving_details
#       Premise
#       Hypothesis

In [ ]:
#answers are in a simple format: problem ID and label pairs
# the header includes configuration info
! head SICK_proving/SICK_trial_pred_50.txt

## FraCaS proving

The LangPro repository also comes with ready `*_sen.pl` and `*_ccg.pl` files for the [FraCaS](https://www-nlp.stanford.edu/~wcmac/downloads/fracas.xml) dataset. This makes easy to run LangPro on FraCaS.

In [ ]:
# Directory where FraCaS proving judgements will be written
! mkdir FraCaS_proving

In [ ]:
# Proving the first 80 FraCaS problems which represents the generalized quantifier section.
# rule application limit is set to 400 as the fracas problems iften have several premises
# allInt is not a useful flag for FraCaS problems as there are many non-itresective adjectives
# fracFilter ignores ill-formed fracas problems (that have no gold label)
! swipl -g "parList([prprb, ral(400), fracFilter, aall, wn_ant, wn_sim, wn_der, constchk, waif('FraCaS_proving/fracas_50.txt')]), entail_some(1-80), halt" \
-f  LangPro/prolog/main.pl  LangPro/WNProlog/wn.pl   LangPro/ccg_sen_d/fracas_sen.pl   LangPro/ccg_sen_d/fracas_d_ccg.pl